# FoodLens Expanded Taxonomy Audit

This notebook starts the broader-class expansion track. The goal is not to train a larger classifier yet; it is to decide whether the attached external datasets have enough useful, auditable labels to justify a model beyond the current 101 Food-101 classes.

- Keep Food-101 as the benchmark taxonomy.
- Audit external dataset folder labels before merging anything.
- Normalize class names so overlaps are visible.
- Export candidate taxonomy files for the next training notebook.
- Do not overwrite the current 101-class benchmark or product champion artifacts.

## 1. Expansion Contract

FoodLens can predict more than 101 classes only after the label space is explicit.

- **Baseline taxonomy:** Food-101 from `kmader/food41`.
- **Candidate broad datasets:** Kaggle food-classification datasets with image folders.
- **Primary output:** an audited union taxonomy, not a trained model.
- **Training rule:** train a broader classifier only after class names, duplicates, and near-duplicates are reviewed.
- **Benchmark rule:** keep Food-101 evaluation separate so broader class coverage does not hide a regression on the existing benchmark.

The next notebook can use the exported taxonomy report to choose between a broad classifier, food-domain pretraining, or regional-specialist heads.

## 2. Imports And Configuration

- Use only filesystem and tabular tooling; this audit does not need GPU.
- Keep dataset definitions in one config table so new Kaggle sources can be added without rewriting the audit.
- Search recursively because Kaggle datasets use different folder layouts.

In [ ]:
import json
import re
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
RESULTS_DIR = Path("/kaggle/working/results/taxonomy_expansion")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

@dataclass(frozen=True)
class DatasetSpec:
    name: str
    root_candidates: tuple[Path, ...]
    class_container_candidates: tuple[str, ...]
    expected_layout: str

DATASETS = [
    DatasetSpec(
        name="food101",
        root_candidates=(
            Path("/kaggle/input/datasets/kmader/food41"),
            Path("/kaggle/input/food41"),
        ),
        class_container_candidates=("images", ""),
        expected_layout="class folders under images/",
    ),
    DatasetSpec(
        name="food_image_classification",
        root_candidates=(
            Path("/kaggle/input/datasets/harishkumardatalab/food-image-classification-dataset"),
            Path("/kaggle/input/food-image-classification-dataset"),
        ),
        class_container_candidates=("Food Classification dataset", ""),
        expected_layout="class folders under Food Classification dataset/",
    ),
    DatasetSpec(
        name="foodies_ai_challenge",
        root_candidates=(
            Path("/kaggle/input/datasets/jvageesh11/foodies-ai-food-image-classification-challenge"),
            Path("/kaggle/input/foodies-ai-food-image-classification-challenge"),
        ),
        class_container_candidates=("Foodies_Challenge", ""),
        expected_layout="class folders under Foodies_Challenge/",
    ),
]

print("Configured datasets:")
for spec in DATASETS:
    print(f"- {spec.name}: {spec.expected_layout}")

## 3. Label Normalization Helpers

- Normalize labels to lowercase snake case.
- Remove split suffixes and noisy filename fragments.
- Keep the original label next to the normalized label so the audit remains reversible.
- Use conservative matching only; this notebook flags candidates, it does not automatically merge semantic near-duplicates.

In [ ]:
def normalize_label(label: str) -> str:
    label = label.strip().lower()
    label = re.sub(r"[()\[\]{}]", " ", label)
    label = re.sub(r"[^a-z0-9]+", "_", label)
    label = re.sub(r"_+", "_", label).strip("_")
    return label


def resolve_existing_root(spec: DatasetSpec) -> Path | None:
    for root in spec.root_candidates:
        if root.exists():
            return root
    return None


def resolve_class_container(spec: DatasetSpec) -> tuple[Path | None, Path | None]:
    root = resolve_existing_root(spec)
    if root is None:
        return None, None
    for relative in spec.class_container_candidates:
        container = root / relative if relative else root
        if not container.exists():
            continue
        class_dirs = [path for path in container.iterdir() if path.is_dir()]
        if class_dirs:
            return root, container
    return root, None


def count_direct_images(class_dir: Path) -> int:
    return sum(
        1
        for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def audit_dataset_classes(spec: DatasetSpec) -> tuple[pd.DataFrame, dict[str, object]]:
    root, container = resolve_class_container(spec)
    summary = {
        "dataset": spec.name,
        "root": str(root) if root else "",
        "container": str(container) if container else "",
        "root_exists": root is not None,
        "container_exists": container is not None,
        "expected_layout": spec.expected_layout,
    }
    if container is None:
        summary.update({"image_count": 0, "class_count": 0})
        return pd.DataFrame(), summary

    records = []
    for class_dir in sorted(path for path in container.iterdir() if path.is_dir()):
        image_count = count_direct_images(class_dir)
        if image_count == 0:
            continue
        records.append(
            {
                "dataset": spec.name,
                "dataset_root": str(root),
                "class_dir": str(class_dir),
                "original_label": class_dir.name,
                "normalized_label": normalize_label(class_dir.name),
                "image_count": image_count,
            }
        )

    class_counts = pd.DataFrame.from_records(records)
    summary.update(
        {
            "image_count": int(class_counts["image_count"].sum()) if not class_counts.empty else 0,
            "class_count": int(class_counts["normalized_label"].nunique()) if not class_counts.empty else 0,
        }
    )
    return class_counts, summary

print(normalize_label("Chocolate Mousse"))
print(normalize_label("Aloo-matar"))

## 4. Dataset Manifests

- Build one image-level manifest per dataset.
- Export image counts and class counts immediately so failed or missing mounts are obvious.
- Do not assume every Kaggle dataset has a clean train/test split or balanced class distribution.

In [ ]:
class_count_parts = []
summary_rows = []
for spec in DATASETS:
    class_counts, summary = audit_dataset_classes(spec)
    if not class_counts.empty:
        class_count_parts.append(class_counts)
    summary_rows.append(summary)

class_counts_df = (
    pd.concat(class_count_parts, ignore_index=True)
    if class_count_parts
    else pd.DataFrame(
        columns=[
            "dataset",
            "dataset_root",
            "class_dir",
            "original_label",
            "normalized_label",
            "image_count",
        ]
    )
)
dataset_summary_df = pd.DataFrame(summary_rows)

class_counts_df.to_csv(RESULTS_DIR / "class_counts_by_dataset.csv", index=False)
dataset_summary_df.to_csv(RESULTS_DIR / "dataset_summary.csv", index=False)

display(dataset_summary_df)
print(f"Total audited images: {class_counts_df['image_count'].sum():,}")
print(f"Total normalized labels across sources: {class_counts_df['normalized_label'].nunique():,}")

## 5. Class Balance And Coverage

- Class imbalance matters more in an expanded taxonomy than in Food-101 because external datasets may have uneven collection processes.
- Very small classes should be held out from training or grouped until enough samples exist.
- Large classes may need capped sampling so they do not dominate the broader classifier.

In [ ]:
class_summary_df = (
    class_counts_df.groupby("dataset")["image_count"]
    .agg(["count", "min", "median", "mean", "max"])
    .reset_index()
    .rename(columns={"count": "class_count"})
)
class_summary_df.to_csv(RESULTS_DIR / "class_balance_summary.csv", index=False)

display(class_summary_df)
for dataset_name in class_counts_df["dataset"].unique():
    dataset_counts = class_counts_df[class_counts_df["dataset"] == dataset_name]
    print(f"\n{dataset_name}: largest classes")
    display(dataset_counts.sort_values("image_count", ascending=False).head(10))
    print(f"{dataset_name}: smallest classes")
    display(dataset_counts.sort_values("image_count", ascending=True).head(10))

## 6. Overlap With Food-101

- Exact normalized-label overlap can be merged safely only after a visual spot-check.
- Non-overlapping labels are expansion candidates.
- Regional or ingredient-specific labels may be better handled as specialist heads instead of one flat classifier.

In [ ]:
food101_labels = set(
    class_counts_df.loc[class_counts_df["dataset"] == "food101", "normalized_label"].unique()
)

overlap_rows = []
for dataset_name, group in class_counts_df.groupby("dataset"):
    labels = set(group["normalized_label"].unique())
    overlap = labels & food101_labels
    new_labels = labels - food101_labels
    overlap_rows.append(
        {
            "dataset": dataset_name,
            "class_count": len(labels),
            "overlap_with_food101": len(overlap),
            "new_vs_food101": len(new_labels),
            "overlap_ratio": len(overlap) / len(labels) if labels else 0.0,
        }
    )

overlap_summary_df = pd.DataFrame(overlap_rows).sort_values("new_vs_food101", ascending=False)
overlap_summary_df.to_csv(RESULTS_DIR / "food101_overlap_summary.csv", index=False)
display(overlap_summary_df)

new_label_df = (
    class_counts_df[~class_counts_df["normalized_label"].isin(food101_labels)]
    .sort_values(["dataset", "image_count"], ascending=[True, False])
)
new_label_df.to_csv(RESULTS_DIR / "new_labels_vs_food101.csv", index=False)

display(new_label_df.head(40))

## 7. Candidate Expanded Taxonomy

- Start with Food-101 labels as stable benchmark labels.
- Add external labels only when they have enough examples.
- Keep a `source_datasets` column so future training can balance classes by source.
- This is a candidate taxonomy, not the final product ontology.

In [ ]:
MIN_IMAGES_FOR_TRAINING = 100

candidate_label_df = (
    class_counts_df.groupby("normalized_label")
    .agg(
        original_labels=("original_label", lambda values: sorted(set(values))),
        source_datasets=("dataset", lambda values: sorted(set(values))),
        total_images=("image_count", "sum"),
    )
    .reset_index()
)
candidate_label_df["in_food101"] = candidate_label_df["normalized_label"].isin(food101_labels)
candidate_label_df["meets_min_images"] = candidate_label_df["total_images"] >= MIN_IMAGES_FOR_TRAINING
candidate_label_df["taxonomy_action"] = candidate_label_df.apply(
    lambda row: "keep_food101_benchmark"
    if row["in_food101"]
    else ("candidate_new_class" if row["meets_min_images"] else "hold_for_more_data"),
    axis=1,
)
candidate_label_df = candidate_label_df.sort_values(
    ["taxonomy_action", "total_images", "normalized_label"], ascending=[True, False, True]
)
candidate_label_df.to_csv(RESULTS_DIR / "candidate_expanded_taxonomy.csv", index=False)

display(candidate_label_df["taxonomy_action"].value_counts().rename_axis("action").reset_index(name="class_count"))
display(candidate_label_df.head(50))

## 8. Recommendation For The Next Training Notebook

Use this audit to choose the next run:

- If one external dataset has many clean new labels, train an expanded classifier head from the A3b checkpoint.
- If labels are regional or imbalanced, use the dataset for food-domain pretraining, then fine-tune back on Food-101.
- If overlap is high but labels are noisy, prioritize decision-layer recalibration and hard-class/noise-aware training first.
- Keep Food-101 metrics as a regression benchmark even when the new classifier predicts more classes.

In [ ]:
recommendation = {
    "baseline_checkpoint": "A3b ConvNeXt-Tiny continued fine-tune",
    "current_food101_test_top_1": 83.90098810195923,
    "current_food101_test_top_5": 95.7821786403656,
    "current_food101_ece_calibrated": 0.055595677345991135,
    "minimum_images_for_new_training_class": MIN_IMAGES_FOR_TRAINING,
    "recommended_next_step": "review candidate_expanded_taxonomy.csv before training a broader classifier",
    "output_dir": str(RESULTS_DIR),
}
(RESULTS_DIR / "taxonomy_audit_summary.json").write_text(json.dumps(recommendation, indent=2))
print(json.dumps(recommendation, indent=2))
print(f"Outputs written to: {RESULTS_DIR}")